# Análise inicial dos microdados do ENEM 2023

Este notebook começa com uma prévia da base e um inventário de colunas e valores ausentes. O inventário lê o CSV em blocos para evitar carregar o arquivo inteiro (cerca de 1,7 GB) na memória.

In [129]:
from pathlib import Path
import pandas as pd

# Funciona tanto com o diretório de trabalho na raiz do projeto quanto em notebooks/.
RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "dados").exists() and (RAIZ_PROJETO.parent / "dados").exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

caminhos_csv = [
    RAIZ_PROJETO / "dados" / "raw" / "MICRODADOS_ENEM_2023.csv",
    RAIZ_PROJETO / "dados" / "raw" / "DADOS" / "MICRODADOS_ENEM_2023.csv",
]
CAMINHO_CSV = next((caminho for caminho in caminhos_csv if caminho.exists()), None)

if CAMINHO_CSV is None:
    raise FileNotFoundError(
        "Não encontrei MICRODADOS_ENEM_2023.csv em dados/raw/ nem em dados/raw/DADOS/."
    )

print(f"Arquivo encontrado: {CAMINHO_CSV}")


Arquivo encontrado: /Users/joaoigorandrade/Documents/unifor/cienciadedados/ciencia-de-dados-unifor-projeto-2/dados/raw/DADOS/MICRODADOS_ENEM_2023.csv


## Prévia dos registros

Este DataFrame mostra as primeiras 10 linhas; não carrega a base inteira.

In [130]:
df_amostra = pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    nrows=10,
)
print(f"Dimensão da prévia: {df_amostra.shape[0]} linhas × {df_amostra.shape[1]} colunas")
display(df_amostra)

Dimensão da prévia: 10 linhas × 76 colunas


,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,...,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
0,210059085136,2023,14,M,2,1,1,1,17,1,...,C,C,B,B,A,B,B,A,A,B
1,210059527735,2023,12,M,2,1,0,1,16,1,...,B,A,B,B,A,A,C,A,D,B
2,210061103945,2023,6,F,1,1,1,1,0,1,...,B,A,A,B,A,A,A,A,A,B
3,210060214087,2023,2,F,1,3,1,2,0,2,...,A,A,A,B,A,A,D,A,A,B
4,210059980948,2023,3,F,1,3,1,2,0,2,...,A,A,A,B,A,A,B,A,A,A
5,210058061539,2023,6,F,1,3,1,1,0,1,...,B,A,A,B,A,A,C,A,A,B
6,210059855122,2023,11,F,1,3,1,1,12,1,...,B,A,A,B,A,B,B,A,A,B
7,210058387333,2023,11,M,1,3,1,1,12,1,...,B,A,A,A,A,A,B,A,B,B
8,210059085137,2023,5,F,1,2,1,1,1,1,...,B,A,A,B,A,A,C,A,A,B
9,210060801601,2023,11,M,1,1,1,1,8,1,...,B,A,B,C,B,A,C,A,B,B


In [131]:
# Seleção inicial para prever Matemática: perfil como X e NU_NOTA_MT como y.
COLUNA_ALVO_MODELO = "NU_NOTA_MT"
colunas_controle_ou_vazamento = {
    "NU_INSCRICAO",  # usado apenas para controle/partições
    "NU_ANO",        # constante nesta edição
    "TP_STATUS_REDACAO",
    "TP_LINGUA",  # removida da seleção de features
}
prefixos_nao_usados_como_features = (
    "NU_NOTA_",       # notas de todas as áreas e competências
    "TP_PRESENCA_",   # presença não é feature
    "TX_RESPOSTAS_",  # respostas não são features
    "TX_GABARITO_",   # gabaritos não são features
    "CO_PROVA_",      # código de prova fora da seleção inicial
)

if COLUNA_ALVO_MODELO not in df_amostra.columns:
    raise KeyError(f"A coluna {COLUNA_ALVO_MODELO} não foi encontrada.")

# Descrições transcritas do Dicionário de Microdados do ENEM 2023.
descricoes_variaveis = {
    'TP_FAIXA_ETARIA': 'Faixa etária2',
    'TP_SEXO': 'Sexo',
    'TP_ESTADO_CIVIL': 'Estado Civil',
    'TP_COR_RACA': 'Cor/raça',
    'TP_NACIONALIDADE': 'Nacionalidade',
    'TP_ST_CONCLUSAO': 'Situação de conclusão do Ensino Médio',
    'TP_ANO_CONCLUIU': 'Ano de Conclusão do Ensino Médio',
    'TP_ESCOLA': 'Tipo de escola do Ensino Médio',
    'TP_ENSINO': 'Tipo de instituição que concluiu ou concluirá o Ensino Médio',
    'IN_TREINEIRO': 'Indica se o inscrito fez a prova com intuito de apenas treinar seus conhecimentos3',
    'CO_MUNICIPIO_ESC': 'Código do município da escola',
    'NO_MUNICIPIO_ESC': 'Nome do município da escola',
    'CO_UF_ESC': 'Código da Unidade da Federação da escola',
    'SG_UF_ESC': 'Sigla da Unidade da Federação da escola',
    'TP_DEPENDENCIA_ADM_ESC': 'Dependência administrativa (Escola)',
    'TP_LOCALIZACAO_ESC': 'Localização (Escola)',
    'TP_SIT_FUNC_ESC': 'Situação de funcionamento (Escola)',
    'CO_MUNICIPIO_PROVA': 'Código do município da aplicação da prova',
    'NO_MUNICIPIO_PROVA': 'Nome do município da aplicação da prova',
    'CO_UF_PROVA': 'Código da Unidade da Federação da aplicação da prova',
    'SG_UF_PROVA': 'Sigla da Unidade da Federação da aplicação da prova',
    'TP_LINGUA': 'Língua Estrangeira',
    'Q001': 'Até que série seu pai, ou o homem responsável por você, estudou?',
    'Q002': 'Até que série sua mãe, ou a mulher responsável por você, estudou?',
    'Q003': 'A partir da apresentação de algumas ocupações divididas em grupos ordenados, indique o grupo que contempla a ocupação mais próxima da ocupação do seu pai ou do homem responsável por você. (Se ele não estiver trabalhando, escolha uma ocupação pensando no último trabalho dele).',
    'Q004': 'A partir da apresentação de algumas ocupações divididas em grupos ordenados, indique o grupo que contempla a ocupação mais próxima da ocupação da sua mãe ou da mulher responsável por você. (Se ela não estiver trabalhando, escolha uma ocupação pensando no último trabalho dela).',
    'Q005': 'Incluindo você, quantas pessoas moram atualmente em sua residência?',
    'Q006': 'Qual é a renda mensal de sua família? (Some a sua renda com a dos seus familiares.)',
    'Q007': 'Em sua residência trabalha empregado(a) doméstico(a)?',
    'Q008': 'Na sua residência tem banheiro?',
    'Q009': 'Na sua residência tem quartos para dormir?',
    'Q010': 'Na sua residência tem carro?',
    'Q011': 'Na sua residência tem motocicleta?',
    'Q012': 'Na sua residência tem geladeira?',
    'Q013': 'Na sua residência tem freezer (independente ou segunda porta da geladeira)?',
    'Q014': 'Na sua residência tem máquina de lavar roupa? (o tanquinho NÃO deve ser considerado)',
    'Q015': 'Na sua residência tem máquina de secar roupa (independente ou em conjunto com a máquina de lavar roupa)?',
    'Q016': 'Na sua residência tem forno micro-ondas?',
    'Q017': 'Na sua residência tem máquina de lavar louça?',
    'Q018': 'Na sua residência tem aspirador de pó?',
    'Q019': 'Na sua residência tem televisão em cores?',
    'Q020': 'Na sua residência tem aparelho de DVD?',
    'Q021': 'Na sua residência tem TV por assinatura?',
    'Q022': 'Na sua residência tem telefone celular?',
    'Q023': 'Na sua residência tem telefone fixo?',
    'Q024': 'Na sua residência tem computador?',
    'Q025': 'Na sua residência tem acesso à Internet?',
    'NU_NOTA_MT': 'Nota da prova de Matemática',
}
rotulos_categorias = {'TP_FAIXA_ETARIA': {'1': 'Menor de 17 anos',
                     '2': '17 anos',
                     '3': '18 anos',
                     '4': '19 anos',
                     '5': '20 anos',
                     '6': '21 anos',
                     '7': '22 anos',
                     '8': '23 anos',
                     '9': '24 anos',
                     '10': '25 anos',
                     '11': 'Entre 26 e 30 anos',
                     '12': 'Entre 31 e 35 anos',
                     '13': 'Entre 36 e 40 anos',
                     '14': 'Entre 41 e 45 anos',
                     '15': 'Entre 46 e 50 anos',
                     '16': 'Entre 51 e 55 anos',
                     '17': 'Entre 56 e 60 anos',
                     '18': 'Entre 61 e 65 anos',
                     '19': 'Entre 66 e 70 anos',
                     '20': 'Maior de 70 anos'},
 'TP_SEXO': {'M': 'Masculino', 'F': 'Feminino'},
 'TP_ESTADO_CIVIL': {'0': 'Não informado',
                     '1': 'Solteiro(a)',
                     '2': 'Casado(a)/Mora com companheiro(a)',
                     '3': 'Divorciado(a)/Desquitado(a)/Separado(a)',
                     '4': 'Viúvo(a)'},
 'TP_COR_RACA': {'0': 'Não declarado',
                 '1': 'Branca',
                 '2': 'Preta',
                 '3': 'Parda',
                 '4': 'Amarela',
                 '5': 'Indígena',
                 '6': 'Não dispõe da informação'},
 'TP_NACIONALIDADE': {'0': 'Não informado',
                      '1': 'Brasileiro(a)',
                      '2': 'Brasileiro(a) Naturalizado(a)',
                      '3': 'Estrangeiro(a)',
                      '4': 'Brasileiro(a) Nato(a), nascido(a) no exterior'},
 'TP_ST_CONCLUSAO': {'1': 'Já concluí o Ensino Médio',
                     '2': 'Estou cursando e concluirei o Ensino Médio em 2023',
                     '3': 'Estou cursando e concluirei o Ensino Médio após 2023',
                     '4': 'Não concluí e não estou cursando o Ensino Médio'},
 'TP_ANO_CONCLUIU': {'0': 'Não informado',
                     '1': '2022',
                     '2': '2021',
                     '3': '2020',
                     '4': '2019',
                     '5': '2018',
                     '6': '2017',
                     '7': '2016',
                     '8': '2015',
                     '9': '2014',
                     '10': '2013',
                     '11': '2012',
                     '12': '2011',
                     '13': '2010',
                     '14': '2009',
                     '15': '2008',
                     '16': '2007',
                     '17': 'Antes de 2007'},
 'TP_ESCOLA': {'1': 'Não Respondeu', '2': 'Pública', '3': 'Privada'},
 'TP_ENSINO': {'1': 'Ensino Regular', '2': 'Educação Especial - Modalidade Substitutiva'},
 'IN_TREINEIRO': {'1': 'Sim', '0': 'Não'},
 'TP_DEPENDENCIA_ADM_ESC': {'1': 'Federal', '2': 'Estadual', '3': 'Municipal', '4': 'Privada'},
 'TP_LOCALIZACAO_ESC': {'1': 'Urbana', '2': 'Rural'},
 'TP_SIT_FUNC_ESC': {'1': 'Em atividade',
                     '2': 'Paralisada',
                     '3': 'Extinta',
                     '4': 'Escola extinta em anos anteriores.'},
 'TP_LINGUA': {'0': 'Inglês', '1': 'Espanhol'},
 'Q001': {'A': 'Nunca estudou.',
          'B': 'Não completou a 4ª série/5º ano do Ensino Fundamental.',
          'C': 'Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino '
               'Fundamental.',
          'D': 'Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino '
               'Médio.',
          'E': 'Completou o Ensino Médio, mas não completou a Faculdade.',
          'F': 'Completou a Faculdade, mas não completou a Pós-graduação.',
          'G': 'Completou a Pós-graduação.',
          'H': 'Não sei.'},
 'Q002': {'A': 'Nunca estudou.',
          'B': 'Não completou a 4ª série/5º ano do Ensino Fundamental.',
          'C': 'Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino '
               'Fundamental.',
          'D': 'Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino '
               'Médio.',
          'E': 'Completou o Ensino Médio, mas não completou a Faculdade.',
          'F': 'Completou a Faculdade, mas não completou a Pós-graduação.',
          'G': 'Completou a Pós-graduação.',
          'H': 'Não sei.'},
 'Q003': {'A': 'Grupo 1: Lavrador, agricultor sem empregados, bóia fria, criador de animais (gado, '
               'porcos, galinhas, ovelhas, cavalos etc.), apicultor, pescador, lenhador, '
               'seringueiro, extrativista.',
          'B': 'Grupo 2: Diarista, empregado doméstico, cuidador de idosos, babá, cozinheiro (em '
               'casas particulares), motorista particular, jardineiro, faxineiro de empresas e '
               'prédios, vigilante, porteiro, carteiro, office-boy, vendedor, caixa, atendente de '
               'loja, auxiliar administrativo, recepcionista, servente de pedreiro, repositor de '
               'mercadoria.',
          'C': 'Grupo 3: Padeiro, cozinheiro industrial ou em restaurantes, sapateiro, costureiro, '
               'joalheiro, torneiro mecânico, operador de máquinas, soldador, operário de fábrica, '
               'trabalhador da mineração, pedreiro, pintor, eletricista, encanador, motorista, '
               'caminhoneiro, taxista.',
          'D': 'Grupo 4: Professor (de ensino fundamental ou médio, idioma, música, artes etc.), '
               'técnico (de enfermagem, contabilidade, eletrônica etc.), policial, militar de '
               'baixa patente (soldado, cabo, sargento), corretor de imóveis, supervisor, gerente, '
               'mestre de obras, pastor, microempresário (proprietário de empresa com menos de 10 '
               'empregados), pequeno comerciante, pequeno proprietário de terras, trabalhador '
               'autônomo ou por conta própria.',
          'E': 'Grupo 5: Médico, engenheiro, dentista, psicólogo, economista, advogado, juiz, '
               'promotor, defensor, delegado, tenente, capitão, coronel, professor universitário, '
               'diretor em empresas públicas ou privadas, político, proprietário de empresas com '
               'mais de 10 empregados.',
          'F': 'Não sei.'},
 'Q004': {'A': 'Grupo 1: Lavradora, agricultora sem empregados, bóia fria, criadora de animais '
               '(gado, porcos, galinhas, ovelhas, cavalos etc.), apicultora, pescadora, lenhadora, '
               'seringueira, extrativista.',
          'B': 'Grupo 2: Diarista, empregada doméstica, cuidadora de idosos, babá, cozinheira (em '
               'casas particulares), motorista particular, jardineira, faxineira de empresas e '
               'prédios, vigilante, porteira, carteira, office-boy, vendedora, caixa, atendente de '
               'loja, auxiliar administrativa, recepcionista, servente de pedreiro, repositora de '
               'mercadoria.',
          'C': 'Grupo 3: Padeira, cozinheira industrial ou em restaurantes, sapateira, costureira, '
               'joalheira, torneira mecânica, operadora de máquinas, soldadora, operária de '
               'fábrica, trabalhadora da mineração, pedreira, pintora, eletricista, encanadora, '
               'motorista, caminhoneira, taxista.',
          'D': 'Grupo 4: Professora (de ensino fundamental ou médio, idioma, música, artes etc.), '
               'técnica (de enfermagem, contabilidade, eletrônica etc.), policial, militar de '
               'baixa patente (soldado, cabo, sargento), corretora de imóveis, supervisora, '
               'gerente, mestre de obras, pastora, microempresária (proprietária de empresa com '
               'menos de 10 empregados), pequena comerciante, pequena proprietária de terras, '
               'trabalhadora autônoma ou por conta própria.',
          'E': 'Grupo 5: Médica, engenheira, dentista, psicóloga, economista, advogada, juíza, '
               'promotora, defensora, delegada, tenente, capitã, coronel, professora '
               'universitária, diretora em empresas públicas ou privadas, política, proprietária '
               'de empresas com mais de 10 empregados.',
          'F': 'Não sei.'},
 'Q005': {'1': '1, pois moro sozinho(a).',
          '2': '2',
          '3': '3',
          '4': '4',
          '5': '5',
          '6': '6',
          '7': '7',
          '8': '8',
          '9': '9',
          '10': '10',
          '11': '11',
          '12': '12',
          '13': '13',
          '14': '14',
          '15': '15',
          '16': '16',
          '17': '17',
          '18': '18',
          '19': '19',
          '20': '20'},
 'Q006': {'A': 'Nenhuma Renda',
          'B': 'Até R$ 1.320,00',
          'C': 'De R$ 1.320,01 até R$ 1.980,00.',
          'D': 'De R$ 1.980,01 até R$ 2.640,00.',
          'E': 'De R$ 2.640,01 até R$ 3.300,00.',
          'F': 'De R$ 3.300,01 até R$ 3.960,00.',
          'G': 'De R$ 3.960,01 até R$ 5.280,00.',
          'H': 'De R$ 5.280,01 até R$ 6.600,00.',
          'I': 'De R$ 6.600,01 até R$ 7.920,00.',
          'J': 'De R$ 7.920,01 até R$ 9240,00.',
          'K': 'De R$ 9.240,01 até R$ 10.560,00.',
          'L': 'De R$ 10.560,01 até R$ 11.880,00.',
          'M': 'De R$ 11.880,01 até R$ 13.200,00.',
          'N': 'De R$ 13.200,01 até R$ 15.840,00.',
          'O': 'De R$ 15.840,01 até R$19.800,00.',
          'P': 'De R$ 19.800,01 até R$ 26.400,00.',
          'Q': 'Acima de R$ 26.400,00.'},
 'Q007': {'A': 'Não.',
          'B': 'Sim, um ou dois dias por semana.',
          'C': 'Sim, três ou quatro dias por semana.',
          'D': 'Sim, pelo menos cinco dias por semana.'},
 'Q008': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q009': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q010': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q011': {'A': 'Não.',
          'B': 'Sim, uma.',
          'C': 'Sim, duas.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q012': {'A': 'Não.',
          'B': 'Sim, uma.',
          'C': 'Sim, duas.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q013': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q014': {'A': 'Não.',
          'B': 'Sim, uma.',
          'C': 'Sim, duas.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q015': {'A': 'Não.',
          'B': 'Sim, uma.',
          'C': 'Sim, duas.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q016': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q017': {'A': 'Não.',
          'B': 'Sim, uma.',
          'C': 'Sim, duas.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q018': {'A': 'Não.', 'B': 'Sim.'},
 'Q019': {'A': 'Não.',
          'B': 'Sim, uma.',
          'C': 'Sim, duas.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q020': {'A': 'Não.', 'B': 'Sim.'},
 'Q021': {'A': 'Não.', 'B': 'Sim.'},
 'Q022': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q023': {'A': 'Não.', 'B': 'Sim.'},
 'Q024': {'A': 'Não.',
          'B': 'Sim, um.',
          'C': 'Sim, dois.',
          'D': 'Sim, três.',
          'E': 'Sim, quatro ou mais.'},
 'Q025': {'A': 'Não.', 'B': 'Sim.'}}


colunas_features_matematica = [
    coluna for coluna in df_amostra.columns
    if coluna != COLUNA_ALVO_MODELO
    and coluna not in colunas_controle_ou_vazamento
    and not coluna.startswith(prefixos_nao_usados_como_features)
]
colunas_selecao_matematica = colunas_features_matematica + [COLUNA_ALVO_MODELO]

def exemplo_da_previa(coluna):
    valores = df_amostra[coluna].dropna()
    return valores.iloc[0] if not valores.empty else "(sem valor nas 10 linhas da prévia)"

# Conta valores distintos não ausentes em toda a base, lendo apenas as colunas selecionadas em blocos.
TAMANHO_BLOCO_UNICOS = 100_000
valores_distintos = {coluna: set() for coluna in colunas_selecao_matematica}
contagens_colunas = {coluna: {} for coluna in colunas_selecao_matematica}
contagens_ausentes = {coluna: 0 for coluna in colunas_selecao_matematica}
linhas_lidas_unicos = 0

leitor_unicos = pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=colunas_selecao_matematica,
    dtype="string",
    chunksize=TAMANHO_BLOCO_UNICOS,
)

for numero_bloco, bloco in enumerate(leitor_unicos, start=1):
    linhas_lidas_unicos += len(bloco)
    for coluna in colunas_selecao_matematica:
        serie = bloco[coluna]
        valores_validos = serie.dropna()
        valores_distintos[coluna].update(valores_validos.unique())
        contagens_ausentes[coluna] += int(serie.isna().sum())
        for valor, quantidade in valores_validos.value_counts().items():
            valor = str(valor)
            contagens_colunas[coluna][valor] = contagens_colunas[coluna].get(valor, 0) + int(quantidade)
    if numero_bloco % 10 == 0:
        print(f"Blocos lidos: {numero_bloco} | linhas: {linhas_lidas_unicos:,}")

def normalizar_codigo_categoria(coluna, valor):
    valor = str(valor)
    categorias = rotulos_categorias.get(coluna, {})
    if valor in categorias:
        return valor
    try:
        numero = float(valor)
        if numero.is_integer() and str(int(numero)) in categorias:
            return str(int(numero))
    except (TypeError, ValueError):
        pass
    return valor


def significado_valor(coluna, valor):
    valor = str(valor)
    categorias = rotulos_categorias.get(coluna, {})
    codigo = normalizar_codigo_categoria(coluna, valor)
    if codigo in categorias:
        return categorias[codigo]
    try:
        float(valor)
        return f"Valor/código numérico de {descricoes_variaveis.get(coluna, coluna)}"
    except ValueError:
        return valor


def plotar_distribuicao_coluna(coluna, limite_categorias_grafico=20):
    """Mostra labels, significados e frequências; no gráfico usa nomes legíveis."""
    frequencias = contagens_colunas[coluna]
    ausentes = contagens_ausentes.get(coluna, 0)

    if coluna == COLUNA_ALVO_MODELO:
        valores_numericos = pd.to_numeric(pd.Series(list(frequencias.keys())), errors="coerce")
        frequencias_nota = pd.Series(list(frequencias.values()), dtype="int64")
        df_resumo_coluna = pd.DataFrame({
            "indicador": ["menor nota", "maior nota", "valores distintos observados", "registros com nota", "registros sem nota"],
            "valor": [
                valores_numericos.min(),
                valores_numericos.max(),
                valores_numericos.nunique(),
                int(frequencias_nota.sum()),
                ausentes,
            ],
        })
        print(f"Resumo de {coluna} (variável numérica contínua):")
        display(df_resumo_coluna)

        df_notas = pd.DataFrame({"nota": valores_numericos, "quantidade_registros": frequencias_nota})
        df_notas["intervalo"] = pd.cut(df_notas["nota"], bins=20, include_lowest=True)
        df_grafico = (
            df_notas.groupby("intervalo", observed=False)["quantidade_registros"]
            .sum()
            .reset_index()
        )
        df_grafico["label_grafico"] = df_grafico["intervalo"].astype(str)
        df_grafico["label"] = df_grafico["label_grafico"]
        df_grafico["significado"] = "Intervalo da nota"
        rotulo_x = "Intervalo da nota"
    else:
        categorias = rotulos_categorias.get(coluna, {})
        linhas_tabela = []
        if categorias:
            # Inclui todas as categorias do dicionário, mesmo as que tiveram contagem zero.
            for codigo, significado in categorias.items():
                codigo_normalizado = normalizar_codigo_categoria(coluna, codigo)
                quantidade = frequencias.get(codigo_normalizado, 0)
                linhas_tabela.append({
                    "label": codigo,
                    "significado": significado,
                    "quantidade_registros": quantidade,
                })
            codigos_documentados = {normalizar_codigo_categoria(coluna, codigo) for codigo in categorias}
            for valor, quantidade in frequencias.items():
                valor_normalizado = normalizar_codigo_categoria(coluna, valor)
                if valor_normalizado not in codigos_documentados:
                    linhas_tabela.append({
                        "label": valor,
                        "significado": "Valor observado não listado no dicionário",
                        "quantidade_registros": quantidade,
                    })
        else:
            # Para colunas sem lista categórica oficial, o valor observado é mostrado como label.
            for valor, quantidade in sorted(frequencias.items(), key=lambda item: item[1], reverse=True):
                linhas_tabela.append({
                    "label": valor,
                    "significado": significado_valor(coluna, valor),
                    "quantidade_registros": quantidade,
                })
        if ausentes:
            linhas_tabela.append({
                "label": "(Ausente)",
                "significado": "Sem valor informado",
                "quantidade_registros": ausentes,
            })
        df_labels_coluna = pd.DataFrame(linhas_tabela)
        print(f"{coluna}: {len(frequencias)} labels observados; ausentes: {ausentes:,}")
        display(df_labels_coluna)
        if coluna.startswith("Q") and len(coluna) == 4 and coluna[1:].isdigit():
            # Destaca as alternativas com maior e menor frequência para cada Q001–Q025.
            df_respostas_validas = df_labels_coluna[df_labels_coluna["label"] != "(Ausente)"]
            maior_frequencia = df_respostas_validas["quantidade_registros"].max()
            menor_frequencia = df_respostas_validas["quantidade_registros"].min()
            print(f"Maior frequência em {coluna}: {maior_frequencia:,}")
            display(df_respostas_validas.loc[df_respostas_validas["quantidade_registros"] == maior_frequencia])
            print(f"Menor frequência em {coluna}: {menor_frequencia:,}")
            display(df_respostas_validas.loc[df_respostas_validas["quantidade_registros"] == menor_frequencia])

        # Usa o significado no eixo X quando há rótulo no dicionário; mantém código no hover.
        df_labels_coluna["label_grafico"] = df_labels_coluna.apply(
            lambda linha: (
                linha["significado"][:37] + "…"
                if linha["label"] in categorias and len(linha["significado"]) > 38
                else linha["significado"] if linha["label"] in categorias
                else linha["label"]
            ),
            axis=1,
        )
        df_grafico = df_labels_coluna[["label_grafico", "label", "significado", "quantidade_registros"]].copy()
        df_grafico = df_grafico.sort_values("quantidade_registros", ascending=False)
        if len(df_grafico) > limite_categorias_grafico:
            df_top = df_grafico.head(limite_categorias_grafico).copy()
            quantidade_outros = int(df_grafico.iloc[limite_categorias_grafico:]["quantidade_registros"].sum())
            df_top.loc[len(df_top)] = ["Outros", "", "Soma das demais categorias", quantidade_outros]
            df_grafico = df_top
        rotulo_x = coluna

    fig_coluna = px.bar(
        df_grafico,
        x="label_grafico",
        y="quantidade_registros",
        custom_data=["label", "significado"],
        title=f"Distribuição de {coluna}",
        labels={"label_grafico": rotulo_x, "quantidade_registros": "Quantidade de registros"},
    )
    fig_coluna.update_traces(
        marker_color="#4C78A8",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Registros: %{y:,.0f}<br>"
            "Label/código: %{customdata[0]}<br>"
            "Significado: %{customdata[1]}<extra></extra>"
        ),
    )
    fig_coluna.update_layout(
        dragmode="select",
        clickmode="event+select",
        xaxis_tickangle=-45,
        template="plotly_white",
    )
    fig_coluna.show()

df_colunas_modelagem_matematica = pd.DataFrame({
    "ordem": range(1, len(colunas_selecao_matematica) + 1),
    "nome_da_coluna": colunas_selecao_matematica,
    "exemplo": [exemplo_da_previa(coluna) for coluna in colunas_selecao_matematica],
    "qtd_valores_distintos": [len(valores_distintos[coluna]) for coluna in colunas_selecao_matematica],
    "descricao": [descricoes_variaveis[coluna] for coluna in colunas_selecao_matematica],
})

# Prévia filtrada de 10 linhas; df_amostra original não é alterado.
df_amostra_modelagem_matematica = df_amostra.loc[:, colunas_selecao_matematica]
print(f"Features de perfil: {len(colunas_features_matematica)} | Alvo: {COLUNA_ALVO_MODELO}")
display(df_colunas_modelagem_matematica)

Blocos lidos: 10 | linhas: 1,000,000
Blocos lidos: 20 | linhas: 2,000,000
Blocos lidos: 30 | linhas: 3,000,000
Blocos lidos: 40 | linhas: 3,933,955
Features de perfil: 46 | Alvo: NU_NOTA_MT


,ordem,nome_da_coluna,exemplo,qtd_valores_distintos,descricao
0,1,TP_FAIXA_ETARIA,14,20,Faixa etária2
1,2,TP_SEXO,M,2,Sexo
2,3,TP_ESTADO_CIVIL,2,5,Estado Civil
3,4,TP_COR_RACA,1,6,Cor/raça
4,5,TP_NACIONALIDADE,1,5,Nacionalidade
5,6,TP_ST_CONCLUSAO,1,4,Situação de conclusão do Ensino Médio
6,7,TP_ANO_CONCLUIU,17,18,Ano de Conclusão do Ensino Médio
7,8,TP_ESCOLA,1,3,Tipo de escola do Ensino Médio
8,9,TP_ENSINO,1.0,2,Tipo de instituição que concluiu ou concluirá ...
9,10,IN_TREINEIRO,0,2,Indica se o inscrito fez a prova com intuito d...


## Descrição das colunas selecionadas

As descrições abaixo correspondem às colunas de perfil e ao alvo de Matemática, com base no Dicionário de Microdados do ENEM 2023.

### `TP_FAIXA_ETARIA`

Faixa etária

In [132]:
# Percorre apenas TP_FAIXA_ETARIA em blocos, sem carregar o CSV inteiro.
contagem_faixa_etaria = {}
ausentes_faixa_etaria = 0

for bloco_faixa in pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=["TP_FAIXA_ETARIA"],
    chunksize=100_000,
):
    valores_bloco = bloco_faixa["TP_FAIXA_ETARIA"].dropna()
    for codigo, quantidade in valores_bloco.value_counts().items():
        codigo = int(codigo)
        contagem_faixa_etaria[codigo] = contagem_faixa_etaria.get(codigo, 0) + int(quantidade)
    ausentes_faixa_etaria += int(bloco_faixa["TP_FAIXA_ETARIA"].isna().sum())

valores_faixa_etaria = sorted(contagem_faixa_etaria)
# Rótulos conforme o Dicionário de Microdados do ENEM 2023.
descricoes_faixa_etaria = {
    1: "Menor de 17 anos",
    2: "17 anos",
    3: "18 anos",
    4: "19 anos",
    5: "20 anos",
    6: "21 anos",
    7: "22 anos",
    8: "23 anos",
    9: "24 anos",
    10: "25 anos",
    11: "Entre 26 e 30 anos",
    12: "Entre 31 e 35 anos",
    13: "Entre 36 e 40 anos",
    14: "Entre 41 e 45 anos",
    15: "Entre 46 e 50 anos",
    16: "Entre 51 e 55 anos",
    17: "Entre 56 e 60 anos",
    18: "Entre 61 e 65 anos",
    19: "Entre 66 e 70 anos",
    20: "Maior de 70 anos",
}
df_faixas_etarias = pd.DataFrame({
    "TP_FAIXA_ETARIA": valores_faixa_etaria,
    "descricao": [descricoes_faixa_etaria.get(valor, "Descrição não encontrada") for valor in valores_faixa_etaria],
    "quantidade_registros": [contagem_faixa_etaria[valor] for valor in valores_faixa_etaria],
})
print(f"Valores distintos encontrados ({len(valores_faixa_etaria)}): {valores_faixa_etaria}")
print(f"Registros sem faixa etária: {ausentes_faixa_etaria:,}")
display(df_faixas_etarias)

Valores distintos encontrados (20): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Registros sem faixa etária: 0


,TP_FAIXA_ETARIA,descricao,quantidade_registros
0,1,Menor de 17 anos,347434
1,2,17 anos,753800
2,3,18 anos,905047
3,4,19 anos,431592
4,5,20 anos,267383
5,6,21 anos,183401
6,7,22 anos,137884
7,8,23 anos,111813
8,9,24 anos,91359
9,10,25 anos,73127


In [133]:
df_faixas_etarias_ordenadas = df_faixas_etarias.sort_values(
    by="quantidade_registros",
    ascending=False,
    ignore_index=True,
)
display(df_faixas_etarias_ordenadas)

,TP_FAIXA_ETARIA,descricao,quantidade_registros
0,3,18 anos,905047
1,2,17 anos,753800
2,4,19 anos,431592
3,1,Menor de 17 anos,347434
4,5,20 anos,267383
5,11,Entre 26 e 30 anos,246292
6,6,21 anos,183401
7,7,22 anos,137884
8,12,Entre 31 e 35 anos,133381
9,8,23 anos,111813


In [134]:
import plotly.express as px

faixas_para_grafico = df_faixas_etarias.sort_values("TP_FAIXA_ETARIA")
fig = px.bar(
    faixas_para_grafico,
    x="descricao",
    y="quantidade_registros",
    custom_data=["TP_FAIXA_ETARIA"],
    labels={
        "descricao": "Faixa etária",
        "quantidade_registros": "Quantidade de registros",
    },
    title="Quantidade de participantes por faixa etária — ENEM 2023",
)
fig.update_traces(
    marker_color="#4C78A8",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Registros: %{y:,.0f}<br>"
        "Código: %{customdata[0]}<extra></extra>"
    ),
)
fig.update_layout(
    dragmode="select",
    clickmode="event+select",
    xaxis_tickangle=-45,
    template="plotly_white",
)
fig.show()

In [135]:
maior_quantidade = df_faixas_etarias["quantidade_registros"].max()
menor_quantidade = df_faixas_etarias["quantidade_registros"].min()

faixas_com_maior_quantidade = df_faixas_etarias.loc[
    df_faixas_etarias["quantidade_registros"] == maior_quantidade
]
faixas_com_menor_quantidade = df_faixas_etarias.loc[
    df_faixas_etarias["quantidade_registros"] == menor_quantidade
]

print(f"Maior quantidade de registros: {maior_quantidade:,}")
display(faixas_com_maior_quantidade)
print(f"Menor quantidade de registros: {menor_quantidade:,}")
display(faixas_com_menor_quantidade)

Maior quantidade de registros: 905,047


,TP_FAIXA_ETARIA,descricao,quantidade_registros
2,3,18 anos,905047


Menor quantidade de registros: 867


,TP_FAIXA_ETARIA,descricao,quantidade_registros
19,20,Maior de 70 anos,867


### `TP_SEXO`

Sexo

In [136]:
# Conta as categorias de TP_SEXO na base completa, em blocos.
contagem_sexo = {}
ausentes_sexo = 0

for bloco_sexo in pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=["TP_SEXO"],
    dtype="string",
    chunksize=100_000,
):
    valores_bloco = bloco_sexo["TP_SEXO"].dropna()
    for codigo, quantidade in valores_bloco.value_counts().items():
        codigo = str(codigo)
        contagem_sexo[codigo] = contagem_sexo.get(codigo, 0) + int(quantidade)
    ausentes_sexo += int(bloco_sexo["TP_SEXO"].isna().sum())

descricoes_sexo = {"M": "Masculino", "F": "Feminino"}
valores_sexo = sorted(contagem_sexo)
df_sexo = pd.DataFrame({
    "TP_SEXO": valores_sexo,
    "descricao": [descricoes_sexo.get(valor, "Descrição não encontrada") for valor in valores_sexo],
    "quantidade_registros": [contagem_sexo[valor] for valor in valores_sexo],
})
print(f"Valores distintos encontrados ({len(valores_sexo)}): {valores_sexo}")
print(f"Registros sem sexo informado: {ausentes_sexo:,}")
display(df_sexo)

Valores distintos encontrados (2): ['F', 'M']
Registros sem sexo informado: 0


,TP_SEXO,descricao,quantidade_registros
0,F,Feminino,2411185
1,M,Masculino,1522770


In [137]:
df_sexo_ordenado = df_sexo.sort_values(
    by="quantidade_registros",
    ascending=False,
    ignore_index=True,
)
display(df_sexo_ordenado)

,TP_SEXO,descricao,quantidade_registros
0,F,Feminino,2411185
1,M,Masculino,1522770


In [138]:
maior_quantidade_sexo = df_sexo["quantidade_registros"].max()
menor_quantidade_sexo = df_sexo["quantidade_registros"].min()

print(f"Maior quantidade de registros: {maior_quantidade_sexo:,}")
display(df_sexo.loc[df_sexo["quantidade_registros"] == maior_quantidade_sexo])
print(f"Menor quantidade de registros: {menor_quantidade_sexo:,}")
display(df_sexo.loc[df_sexo["quantidade_registros"] == menor_quantidade_sexo])

Maior quantidade de registros: 2,411,185


,TP_SEXO,descricao,quantidade_registros
0,F,Feminino,2411185


Menor quantidade de registros: 1,522,770


,TP_SEXO,descricao,quantidade_registros
1,M,Masculino,1522770


In [139]:
import plotly.express as px

fig_sexo = px.bar(
    df_sexo_ordenado,
    x="descricao",
    y="quantidade_registros",
    custom_data=["TP_SEXO"],
    labels={
        "descricao": "Sexo",
        "quantidade_registros": "Quantidade de registros",
    },
    title="Quantidade de participantes por sexo — ENEM 2023",
)
fig_sexo.update_traces(
    marker_color="#4C78A8",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Registros: %{y:,.0f}<br>"
        "Código: %{customdata[0]}<extra></extra>"
    ),
)
fig_sexo.update_layout(
    dragmode="select",
    clickmode="event+select",
    template="plotly_white",
)
fig_sexo.show()

In [140]:
# Relação descritiva entre categorias: conta pares faixa etária × sexo na base completa.
contagem_sexo_faixa = {}
registros_sem_sexo_ou_faixa = 0

for bloco_sexo_faixa in pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=["TP_FAIXA_ETARIA", "TP_SEXO"],
    dtype={"TP_SEXO": "string"},
    chunksize=100_000,
):
    registros_sem_sexo_ou_faixa += int(
        bloco_sexo_faixa[["TP_FAIXA_ETARIA", "TP_SEXO"]].isna().any(axis=1).sum()
    )
    pares_bloco = (
        bloco_sexo_faixa
        .dropna(subset=["TP_FAIXA_ETARIA", "TP_SEXO"])
        .groupby(["TP_FAIXA_ETARIA", "TP_SEXO"], observed=True)
        .size()
    )
    for (faixa, sexo), quantidade in pares_bloco.items():
        chave = (int(faixa), str(sexo))
        contagem_sexo_faixa[chave] = contagem_sexo_faixa.get(chave, 0) + int(quantidade)

linhas_sexo_faixa = [
    {
        "codigo_faixa_etaria": faixa,
        "faixa_etaria": descricoes_faixa_etaria[faixa],
        "codigo_sexo": sexo,
        "sexo": descricoes_sexo.get(sexo, sexo),
        "quantidade_registros": quantidade,
    }
    for (faixa, sexo), quantidade in contagem_sexo_faixa.items()
]
df_sexo_por_faixa = pd.DataFrame(linhas_sexo_faixa)
df_sexo_por_faixa["total_na_faixa"] = df_sexo_por_faixa.groupby("codigo_faixa_etaria")["quantidade_registros"].transform("sum")
df_sexo_por_faixa["percentual_na_faixa"] = (
    df_sexo_por_faixa["quantidade_registros"] / df_sexo_por_faixa["total_na_faixa"] * 100
).round(1)

print(f"Registros sem sexo ou faixa etária (excluídos do cruzamento): {registros_sem_sexo_ou_faixa:,}")

fig_sexo_por_faixa = px.bar(
    df_sexo_por_faixa,
    x="faixa_etaria",
    y="quantidade_registros",
    color="sexo",
    barmode="group",
    custom_data=["codigo_faixa_etaria", "codigo_sexo", "percentual_na_faixa"],
    category_orders={
        "faixa_etaria": [descricoes_faixa_etaria[codigo] for codigo in sorted(descricoes_faixa_etaria)],
        "sexo": ["Feminino", "Masculino"],
    },
    labels={
        "faixa_etaria": "Faixa etária",
        "quantidade_registros": "Quantidade de registros",
        "sexo": "Sexo",
    },
    title="Distribuição de participantes por sexo e faixa etária — ENEM 2023",
)
fig_sexo_por_faixa.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Sexo: %{fullData.name}<br>"
        "Registros: %{y:,.0f}<br>"
        "Dentro da faixa etária: %{customdata[2]:.1f}%<extra></extra>"
    ),
)
fig_sexo_por_faixa.update_layout(
    dragmode="select",
    clickmode="event+select",
    xaxis_tickangle=-45,
    template="plotly_white",
)
fig_sexo_por_faixa.show()

Registros sem sexo ou faixa etária (excluídos do cruzamento): 0


### `TP_ESTADO_CIVIL`

Estado Civil

In [141]:
# Conta as categorias de TP_ESTADO_CIVIL na base completa, em blocos.
contagem_estado_civil = {}
ausentes_estado_civil = 0

for bloco_estado_civil in pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=["TP_ESTADO_CIVIL"],
    dtype="string",
    chunksize=100_000,
):
    valores_bloco = bloco_estado_civil["TP_ESTADO_CIVIL"].dropna()
    for codigo, quantidade in valores_bloco.value_counts().items():
        codigo = int(codigo)
        contagem_estado_civil[codigo] = contagem_estado_civil.get(codigo, 0) + int(quantidade)
    ausentes_estado_civil += int(bloco_estado_civil["TP_ESTADO_CIVIL"].isna().sum())

descricoes_estado_civil = {
    0: "Não informado",
    1: "Solteiro(a)",
    2: "Casado(a)/Mora com companheiro(a)",
    3: "Divorciado(a)/Desquitado(a)/Separado(a)",
    4: "Viúvo(a)",
}
valores_estado_civil = sorted(contagem_estado_civil)
df_estado_civil = pd.DataFrame({
    "TP_ESTADO_CIVIL": valores_estado_civil,
    "descricao": [descricoes_estado_civil.get(valor, "Descrição não encontrada") for valor in valores_estado_civil],
    "quantidade_registros": [contagem_estado_civil[valor] for valor in valores_estado_civil],
})
print(f"Valores distintos encontrados ({len(valores_estado_civil)}): {valores_estado_civil}")
print(f"Registros sem estado civil informado (valor ausente): {ausentes_estado_civil:,}")
display(df_estado_civil)

Valores distintos encontrados (5): [0, 1, 2, 3, 4]
Registros sem estado civil informado (valor ausente): 0


,TP_ESTADO_CIVIL,descricao,quantidade_registros
0,0,Não informado,171900
1,1,Solteiro(a),3491857
2,2,Casado(a)/Mora com companheiro(a),200456
3,3,Divorciado(a)/Desquitado(a)/Separado(a),64933
4,4,Viúvo(a),4809


In [142]:
df_estado_civil_ordenado = df_estado_civil.sort_values(
    by="quantidade_registros",
    ascending=False,
    ignore_index=True,
)
display(df_estado_civil_ordenado)

,TP_ESTADO_CIVIL,descricao,quantidade_registros
0,1,Solteiro(a),3491857
1,2,Casado(a)/Mora com companheiro(a),200456
2,0,Não informado,171900
3,3,Divorciado(a)/Desquitado(a)/Separado(a),64933
4,4,Viúvo(a),4809


In [143]:
maior_quantidade_estado_civil = df_estado_civil["quantidade_registros"].max()
menor_quantidade_estado_civil = df_estado_civil["quantidade_registros"].min()

print(f"Maior quantidade de registros: {maior_quantidade_estado_civil:,}")
display(df_estado_civil.loc[df_estado_civil["quantidade_registros"] == maior_quantidade_estado_civil])
print(f"Menor quantidade de registros: {menor_quantidade_estado_civil:,}")
display(df_estado_civil.loc[df_estado_civil["quantidade_registros"] == menor_quantidade_estado_civil])

Maior quantidade de registros: 3,491,857


,TP_ESTADO_CIVIL,descricao,quantidade_registros
1,1,Solteiro(a),3491857


Menor quantidade de registros: 4,809


,TP_ESTADO_CIVIL,descricao,quantidade_registros
4,4,Viúvo(a),4809


In [144]:
import plotly.express as px

fig_estado_civil = px.bar(
    df_estado_civil,
    x="descricao",
    y="quantidade_registros",
    custom_data=["TP_ESTADO_CIVIL"],
    category_orders={
        "descricao": [descricoes_estado_civil[codigo] for codigo in sorted(descricoes_estado_civil)],
    },
    labels={
        "descricao": "Estado civil",
        "quantidade_registros": "Quantidade de registros",
    },
    title="Quantidade de participantes por estado civil — ENEM 2023",
)
fig_estado_civil.update_traces(
    marker_color="#4C78A8",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Registros: %{y:,.0f}<br>"
        "Código: %{customdata[0]}<extra></extra>"
    ),
)
fig_estado_civil.update_layout(
    dragmode="select",
    clickmode="event+select",
    xaxis_tickangle=-30,
    template="plotly_white",
)
fig_estado_civil.show()

In [145]:
# Conta os cruzamentos na base completa em uma única leitura por blocos.
contagem_faixa_estado_civil = {}
contagem_estado_civil_sexo = {}
sem_faixa_estado_civil = 0
sem_estado_civil_sexo = 0

for bloco_cruzamentos in pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=["TP_FAIXA_ETARIA", "TP_ESTADO_CIVIL", "TP_SEXO"],
    dtype={"TP_ESTADO_CIVIL": "string", "TP_SEXO": "string"},
    chunksize=100_000,
):
    sem_faixa_estado_civil += int(
        bloco_cruzamentos[["TP_FAIXA_ETARIA", "TP_ESTADO_CIVIL"]].isna().any(axis=1).sum()
    )
    pares_faixa_civil = (
        bloco_cruzamentos
        .dropna(subset=["TP_FAIXA_ETARIA", "TP_ESTADO_CIVIL"])
        .groupby(["TP_FAIXA_ETARIA", "TP_ESTADO_CIVIL"], observed=True)
        .size()
    )
    for (faixa, estado), quantidade in pares_faixa_civil.items():
        chave = (int(faixa), int(estado))
        contagem_faixa_estado_civil[chave] = contagem_faixa_estado_civil.get(chave, 0) + int(quantidade)

    sem_estado_civil_sexo += int(
        bloco_cruzamentos[["TP_ESTADO_CIVIL", "TP_SEXO"]].isna().any(axis=1).sum()
    )
    pares_civil_sexo = (
        bloco_cruzamentos
        .dropna(subset=["TP_ESTADO_CIVIL", "TP_SEXO"])
        .groupby(["TP_ESTADO_CIVIL", "TP_SEXO"], observed=True)
        .size()
    )
    for (estado, sexo), quantidade in pares_civil_sexo.items():
        chave = (int(estado), str(sexo))
        contagem_estado_civil_sexo[chave] = contagem_estado_civil_sexo.get(chave, 0) + int(quantidade)

# Matrizes completas: incluem combinações com zero registros.
linhas_faixa_estado_civil = [
    {
        "codigo_faixa": faixa,
        "faixa_etaria": descricoes_faixa_etaria[faixa],
        "codigo_estado_civil": estado,
        "estado_civil": descricoes_estado_civil[estado],
        "quantidade_registros": contagem_faixa_estado_civil.get((faixa, estado), 0),
    }
    for faixa in sorted(descricoes_faixa_etaria)
    for estado in sorted(descricoes_estado_civil)
]
df_faixa_estado_civil = pd.DataFrame(linhas_faixa_estado_civil)
df_faixa_estado_civil["total_faixa"] = df_faixa_estado_civil.groupby("codigo_faixa")["quantidade_registros"].transform("sum")
df_faixa_estado_civil["percentual_na_faixa"] = (
    df_faixa_estado_civil["quantidade_registros"] / df_faixa_estado_civil["total_faixa"] * 100
).fillna(0).round(1)

linhas_estado_civil_sexo = [
    {
        "codigo_estado_civil": estado,
        "estado_civil": descricoes_estado_civil[estado],
        "codigo_sexo": sexo,
        "sexo": descricoes_sexo[sexo],
        "quantidade_registros": contagem_estado_civil_sexo.get((estado, sexo), 0),
    }
    for estado in sorted(descricoes_estado_civil)
    for sexo in sorted(descricoes_sexo)
]
df_estado_civil_sexo = pd.DataFrame(linhas_estado_civil_sexo)
df_estado_civil_sexo["total_estado_civil"] = df_estado_civil_sexo.groupby("codigo_estado_civil")["quantidade_registros"].transform("sum")
df_estado_civil_sexo["percentual_no_estado_civil"] = (
    df_estado_civil_sexo["quantidade_registros"] / df_estado_civil_sexo["total_estado_civil"] * 100
).fillna(0).round(1)

print(f"Faixa etária × estado civil: linhas sem algum dos campos = {sem_faixa_estado_civil:,}")
print(f"Estado civil × sexo: linhas sem algum dos campos = {sem_estado_civil_sexo:,}")

Faixa etária × estado civil: linhas sem algum dos campos = 0
Estado civil × sexo: linhas sem algum dos campos = 0


In [146]:
import plotly.graph_objects as go

ordem_faixas = [descricoes_faixa_etaria[codigo] for codigo in sorted(descricoes_faixa_etaria)]
ordem_estados_civis = [descricoes_estado_civil[codigo] for codigo in sorted(descricoes_estado_civil)]
percentual_faixa_civil = (
    df_faixa_estado_civil
    .pivot(index="faixa_etaria", columns="estado_civil", values="percentual_na_faixa")
    .reindex(index=ordem_faixas, columns=ordem_estados_civis)
    .fillna(0)
)
contagem_faixa_civil = (
    df_faixa_estado_civil
    .pivot(index="faixa_etaria", columns="estado_civil", values="quantidade_registros")
    .reindex(index=ordem_faixas, columns=ordem_estados_civis)
    .fillna(0)
)

fig_faixa_estado_civil = go.Figure(data=go.Heatmap(
    z=percentual_faixa_civil.values,
    x=percentual_faixa_civil.columns,
    y=percentual_faixa_civil.index,
    customdata=contagem_faixa_civil.values,
    colorscale="Blues",
    zmin=0,
    zmax=100,
    colorbar={"title": "% dentro<br>da faixa"},
    hovertemplate=(
        "Faixa etária: %{y}<br>"
        "Estado civil: %{x}<br>"
        "Percentual da faixa: %{z:.1f}%<br>"
        "Registros: %{customdata:,.0f}<extra></extra>"
    ),
))
fig_faixa_estado_civil.update_layout(
    title="Estado civil dentro de cada faixa etária — ENEM 2023",
    xaxis_title="Estado civil",
    yaxis_title="Faixa etária",
    dragmode="select",
    clickmode="event+select",
    template="plotly_white",
    height=750,
)
fig_faixa_estado_civil.show()

In [147]:
percentual_civil_sexo = (
    df_estado_civil_sexo
    .pivot(index="estado_civil", columns="sexo", values="percentual_no_estado_civil")
    .reindex(index=ordem_estados_civis, columns=["Feminino", "Masculino"])
    .fillna(0)
)
contagem_civil_sexo = (
    df_estado_civil_sexo
    .pivot(index="estado_civil", columns="sexo", values="quantidade_registros")
    .reindex(index=ordem_estados_civis, columns=["Feminino", "Masculino"])
    .fillna(0)
)

fig_estado_civil_sexo = go.Figure(data=go.Heatmap(
    z=percentual_civil_sexo.values,
    x=percentual_civil_sexo.columns,
    y=percentual_civil_sexo.index,
    customdata=contagem_civil_sexo.values,
    colorscale="Purples",
    zmin=0,
    zmax=100,
    colorbar={"title": "% dentro<br>do estado civil"},
    hovertemplate=(
        "Estado civil: %{y}<br>"
        "Sexo: %{x}<br>"
        "Percentual do estado civil: %{z:.1f}%<br>"
        "Registros: %{customdata:,.0f}<extra></extra>"
    ),
))
fig_estado_civil_sexo.update_layout(
    title="Sexo dentro de cada categoria de estado civil — ENEM 2023",
    xaxis_title="Sexo",
    yaxis_title="Estado civil",
    dragmode="select",
    clickmode="event+select",
    template="plotly_white",
    height=500,
)
fig_estado_civil_sexo.show()

### `TP_COR_RACA`

Cor/raça

In [148]:
plotar_distribuicao_coluna('TP_COR_RACA')

TP_COR_RACA: 6 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,0,Não declarado,52575
1,1,Branca,1575848
2,2,Preta,509511
3,3,Parda,1706798
4,4,Amarela,64512
5,5,Indígena,24711
6,6,Não dispõe da informação,0


### `TP_NACIONALIDADE`

Nacionalidade

In [149]:
plotar_distribuicao_coluna('TP_NACIONALIDADE')

TP_NACIONALIDADE: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,0,Não informado,2153
1,1,Brasileiro(a),3842681
2,2,Brasileiro(a) Naturalizado(a),73429
3,3,Estrangeiro(a),7112
4,4,"Brasileiro(a) Nato(a), nascido(a) no exterior",8580


### `TP_ST_CONCLUSAO`

Situação de conclusão do Ensino Médio

In [150]:
plotar_distribuicao_coluna('TP_ST_CONCLUSAO')

TP_ST_CONCLUSAO: 4 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,1,Já concluí o Ensino Médio,1895301
1,2,Estou cursando e concluirei o Ensino Médio em ...,1401164
2,3,Estou cursando e concluirei o Ensino Médio apó...,620067
3,4,Não concluí e não estou cursando o Ensino Médio,17423


### `TP_ANO_CONCLUIU`

Ano de Conclusão do Ensino Médio

In [151]:
plotar_distribuicao_coluna('TP_ANO_CONCLUIU')

TP_ANO_CONCLUIU: 18 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,0,Não informado,2243134
1,1,2022,418530
2,2,2021,264183
3,3,2020,149798
4,4,2019,136449
5,5,2018,104195
6,6,2017,85411
7,7,2016,65549
8,8,2015,54769
9,9,2014,46654


### `TP_ESCOLA`

Tipo de escola do Ensino Médio

In [152]:
plotar_distribuicao_coluna('TP_ESCOLA')

TP_ESCOLA: 3 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,1,Não Respondeu,2532796
1,2,Pública,1166540
2,3,Privada,234619


### `TP_ENSINO`

Tipo de instituição que concluiu ou concluirá o Ensino Médio

In [153]:
plotar_distribuicao_coluna('TP_ENSINO')

TP_ENSINO: 2 labels observados; ausentes: 2,594,874


,label,significado,quantidade_registros
0,1,Ensino Regular,1332195
1,2,Educação Especial - Modalidade Substitutiva,6886
2,(Ausente),Sem valor informado,2594874


### `IN_TREINEIRO`

Indica se o inscrito fez a prova com intuito de apenas treinar seus conhecimentos

In [154]:
plotar_distribuicao_coluna('IN_TREINEIRO')

IN_TREINEIRO: 2 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,1,Sim,620067
1,0,Não,3313888


## Relação entre situação de conclusão do Ensino Médio e treineiro

Como as duas variáveis são categóricas, a relação será apresentada por tabela cruzada, percentuais por situação de conclusão e teste qui-quadrado de independência, acompanhado do V de Cramér como medida de associação. Não se usa correlação de Pearson diretamente sobre os códigos.

In [155]:
from scipy.stats import chi2_contingency

colunas_relacao_treineiro = ["TP_ST_CONCLUSAO", "IN_TREINEIRO"]
contagens_relacao_treineiro = {}
ausentes_relacao_treineiro = 0
linhas_relacao_treineiro = 0

for bloco_relacao in pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin1",
    usecols=colunas_relacao_treineiro,
    dtype={coluna: "string" for coluna in colunas_relacao_treineiro},
    chunksize=100_000,
):
    linhas_relacao_treineiro += len(bloco_relacao)
    ausentes_relacao_treineiro += int(bloco_relacao[colunas_relacao_treineiro].isna().any(axis=1).sum())
    pares_relacao = (
        bloco_relacao
        .dropna(subset=colunas_relacao_treineiro)
        .groupby(colunas_relacao_treineiro, observed=True)
        .size()
    )
    for (situacao, treineiro), quantidade in pares_relacao.items():
        chave = (str(situacao), str(treineiro))
        contagens_relacao_treineiro[chave] = contagens_relacao_treineiro.get(chave, 0) + int(quantidade)

rotulos_conclusao = rotulos_categorias["TP_ST_CONCLUSAO"]
rotulos_treineiro = rotulos_categorias["IN_TREINEIRO"]
codigos_conclusao = list(rotulos_conclusao)
codigos_treineiro = list(rotulos_treineiro)

matriz_contagens_treineiro = pd.DataFrame(
    0,
    index=[rotulos_conclusao[codigo] for codigo in codigos_conclusao],
    columns=[rotulos_treineiro[codigo] for codigo in codigos_treineiro],
    dtype="int64",
)
for codigo_conclusao in codigos_conclusao:
    for codigo_treineiro in codigos_treineiro:
        matriz_contagens_treineiro.loc[
            rotulos_conclusao[codigo_conclusao], rotulos_treineiro[codigo_treineiro]
        ] = contagens_relacao_treineiro.get((codigo_conclusao, codigo_treineiro), 0)

matriz_percentuais_treineiro = (
    matriz_contagens_treineiro
    .div(matriz_contagens_treineiro.sum(axis=1).replace(0, pd.NA), axis=0)
    .mul(100)
    .fillna(0)
    .round(2)
)

# O teste é calculado somente nas categorias com registros observados.
matriz_teste_treineiro = matriz_contagens_treineiro.loc[
    matriz_contagens_treineiro.sum(axis=1) > 0,
    matriz_contagens_treineiro.sum(axis=0) > 0,
]
chi2_treineiro, p_treineiro, gl_treineiro, esperados_treineiro = chi2_contingency(matriz_teste_treineiro)
n_treineiro_validos = int(matriz_teste_treineiro.to_numpy().sum())
menor_dimensao_treineiro = min(matriz_teste_treineiro.shape)
v_cramer_treineiro = (chi2_treineiro / (n_treineiro_validos * (menor_dimensao_treineiro - 1))) ** 0.5

print(f"Registros na base: {linhas_relacao_treineiro:,}")
print(f"Registros com ambos os campos preenchidos: {n_treineiro_validos:,}")
print(f"Registros ausentes em pelo menos um dos campos: {ausentes_relacao_treineiro:,}")
print("Contagem de participantes por situação de conclusão × condição de treineiro:")
display(matriz_contagens_treineiro)
print("Percentual de treineiros/não treineiros dentro de cada situação de conclusão:")
display(matriz_percentuais_treineiro)
print(f"Qui-quadrado: {chi2_treineiro:,.2f} | graus de liberdade: {gl_treineiro} | p-valor: {p_treineiro:.3e}")
print(f"V de Cramér: {v_cramer_treineiro:.4f}")


Registros na base: 3,933,955
Registros com ambos os campos preenchidos: 3,933,955
Registros ausentes em pelo menos um dos campos: 0
Contagem de participantes por situação de conclusão × condição de treineiro:


,Sim,Não
Já concluí o Ensino Médio,0,1895301
Estou cursando e concluirei o Ensino Médio em 2023,0,1401164
Estou cursando e concluirei o Ensino Médio após 2023,620067,0
Não concluí e não estou cursando o Ensino Médio,0,17423


Percentual de treineiros/não treineiros dentro de cada situação de conclusão:


,Sim,Não
Já concluí o Ensino Médio,0.0,100.0
Estou cursando e concluirei o Ensino Médio em 2023,0.0,100.0
Estou cursando e concluirei o Ensino Médio após 2023,100.0,0.0
Não concluí e não estou cursando o Ensino Médio,0.0,100.0


Qui-quadrado: 3,933,955.00 | graus de liberdade: 3 | p-valor: 0.000e+00
V de Cramér: 1.0000


In [156]:
import plotly.express as px

linhas_grafico_treineiro = []
for codigo_conclusao in codigos_conclusao:
    nome_conclusao = rotulos_conclusao[codigo_conclusao]
    total_situacao = int(matriz_contagens_treineiro.loc[nome_conclusao].sum())
    for codigo_treineiro in codigos_treineiro:
        nome_treineiro = rotulos_treineiro[codigo_treineiro]
        quantidade = int(matriz_contagens_treineiro.loc[nome_conclusao, nome_treineiro])
        percentual = float(matriz_percentuais_treineiro.loc[nome_conclusao, nome_treineiro])
        linhas_grafico_treineiro.append({
            "situacao_conclusao": nome_conclusao,
            "condicao_treineiro": nome_treineiro,
            "quantidade_registros": quantidade,
            "percentual": percentual,
            "total_situacao": total_situacao,
        })

df_grafico_treineiro = pd.DataFrame(linhas_grafico_treineiro)
fig_treineiro_conclusao = px.bar(
    df_grafico_treineiro,
    x="situacao_conclusao",
    y="percentual",
    color="condicao_treineiro",
    barmode="stack",
    custom_data=["quantidade_registros", "total_situacao"],
    category_orders={
        "situacao_conclusao": [rotulos_conclusao[codigo] for codigo in codigos_conclusao],
        "condicao_treineiro": [rotulos_treineiro[codigo] for codigo in codigos_treineiro],
    },
    title="Condição de treineiro por situação de conclusão do Ensino Médio",
    labels={
        "situacao_conclusao": "Situação de conclusão do Ensino Médio",
        "condicao_treineiro": "Treineiro",
        "percentual": "Participantes dentro da situação (%)",
    },
)
fig_treineiro_conclusao.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Condição: %{fullData.name}<br>"
        "Percentual: %{y:.2f}%<br>"
        "Registros: %{customdata[0]:,.0f} de %{customdata[1]:,.0f}<extra></extra>"
    )
)
fig_treineiro_conclusao.update_layout(
    yaxis={"range": [0, 100], "ticksuffix": "%"},
    template="plotly_white",
    legend_title_text="Fez a prova como treineiro?",
)
fig_treineiro_conclusao.show()


### `CO_MUNICIPIO_ESC`

Código do município da escola

In [157]:
plotar_distribuicao_coluna('CO_MUNICIPIO_ESC')

CO_MUNICIPIO_ESC: 5528 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,3550308,Valor/código numérico de Código do município d...,45215
1,2304400,Valor/código numérico de Código do município d...,27634
2,3304557,Valor/código numérico de Código do município d...,26064
3,5300108,Valor/código numérico de Código do município d...,20113
4,1302603,Valor/código numérico de Código do município d...,15500
...,...,...,...
5524,4314753,Valor/código numérico de Código do município d...,1
5525,4313003,Valor/código numérico de Código do município d...,1
5526,3100708,Valor/código numérico de Código do município d...,1
5527,4322343,Valor/código numérico de Código do município d...,1


### `NO_MUNICIPIO_ESC`

Nome do município da escola

In [158]:
plotar_distribuicao_coluna('NO_MUNICIPIO_ESC')

NO_MUNICIPIO_ESC: 5261 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,São Paulo,São Paulo,45215
1,Fortaleza,Fortaleza,27634
2,Rio de Janeiro,Rio de Janeiro,26064
3,Brasília,Brasília,20113
4,Manaus,Manaus,15500
...,...,...,...
5257,Poço das Antas,Poço das Antas,1
5258,Nova Bréscia,Nova Bréscia,1
5259,Água Comprida,Água Comprida,1
5260,Ubiretama,Ubiretama,1


### `CO_UF_ESC`

Código da Unidade da Federação da escola

In [159]:
plotar_distribuicao_coluna('CO_UF_ESC')

CO_UF_ESC: 27 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,35,Valor/código numérico de Código da Unidade da ...,187067
1,23,Valor/código numérico de Código da Unidade da ...,98595
2,31,Valor/código numérico de Código da Unidade da ...,66972
3,33,Valor/código numérico de Código da Unidade da ...,63842
4,41,Valor/código numérico de Código da Unidade da ...,54226
5,52,Valor/código numérico de Código da Unidade da ...,45796
6,26,Valor/código numérico de Código da Unidade da ...,44881
7,29,Valor/código numérico de Código da Unidade da ...,43453
8,43,Valor/código numérico de Código da Unidade da ...,39700
9,15,Valor/código numérico de Código da Unidade da ...,35212


### `SG_UF_ESC`

Sigla da Unidade da Federação da escola

In [160]:
plotar_distribuicao_coluna('SG_UF_ESC')

SG_UF_ESC: 27 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,SP,SP,187067
1,CE,CE,98595
2,MG,MG,66972
3,RJ,RJ,63842
4,PR,PR,54226
5,GO,GO,45796
6,PE,PE,44881
7,BA,BA,43453
8,RS,RS,39700
9,PA,PA,35212


### `TP_DEPENDENCIA_ADM_ESC`

Dependência administrativa (Escola)

In [161]:
plotar_distribuicao_coluna('TP_DEPENDENCIA_ADM_ESC')

TP_DEPENDENCIA_ADM_ESC: 4 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,1,Federal,48433
1,2,Estadual,671488
2,3,Municipal,8957
3,4,Privada,229628
4,(Ausente),Sem valor informado,2975449


### `TP_LOCALIZACAO_ESC`

Localização (Escola)

In [162]:
plotar_distribuicao_coluna('TP_LOCALIZACAO_ESC')

TP_LOCALIZACAO_ESC: 2 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,1,Urbana,922856
1,2,Rural,35650
2,(Ausente),Sem valor informado,2975449


### `TP_SIT_FUNC_ESC`

Situação de funcionamento (Escola)

In [163]:
plotar_distribuicao_coluna('TP_SIT_FUNC_ESC')

TP_SIT_FUNC_ESC: 4 labels observados; ausentes: 2,975,449


,label,significado,quantidade_registros
0,1,Em atividade,950936
1,2,Paralisada,3182
2,3,Extinta,507
3,4,Escola extinta em anos anteriores.,3881
4,(Ausente),Sem valor informado,2975449


### `CO_MUNICIPIO_PROVA`

Código do município da aplicação da prova

In [164]:
plotar_distribuicao_coluna('CO_MUNICIPIO_PROVA')

CO_MUNICIPIO_PROVA: 1750 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,3550308,Valor/código numérico de Código do município d...,157220
1,3304557,Valor/código numérico de Código do município d...,116997
2,2304400,Valor/código numérico de Código do município d...,74112
3,5300108,Valor/código numérico de Código do município d...,72975
4,2927408,Valor/código numérico de Código do município d...,69514
...,...,...,...
1745,1200328,Valor/código numérico de Código do município d...,66
1746,1302108,Valor/código numérico de Código do município d...,65
1747,1400159,Valor/código numérico de Código do município d...,51
1748,1200435,Valor/código numérico de Código do município d...,41


### `NO_MUNICIPIO_PROVA`

Nome do município da aplicação da prova

In [165]:
plotar_distribuicao_coluna('NO_MUNICIPIO_PROVA')

NO_MUNICIPIO_PROVA: 1715 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,São Paulo,São Paulo,157220
1,Rio de Janeiro,Rio de Janeiro,116997
2,Fortaleza,Fortaleza,74112
3,Brasília,Brasília,72975
4,Salvador,Salvador,69514
...,...,...,...
1710,Jordão,Jordão,66
1711,Japurá,Japurá,65
1712,Bonfim,Bonfim,51
1713,Santa Rosa do Purus,Santa Rosa do Purus,41


### `CO_UF_PROVA`

Código da Unidade da Federação da aplicação da prova

In [166]:
plotar_distribuicao_coluna('CO_UF_PROVA')

CO_UF_PROVA: 27 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,35,Valor/código numérico de Código da Unidade da ...,590767
1,31,Valor/código numérico de Código da Unidade da ...,358575
2,29,Valor/código numérico de Código da Unidade da ...,324268
3,33,Valor/código numérico de Código da Unidade da ...,282296
4,23,Valor/código numérico de Código da Unidade da ...,241960
5,15,Valor/código numérico de Código da Unidade da ...,229162
6,26,Valor/código numérico de Código da Unidade da ...,218859
7,41,Valor/código numérico de Código da Unidade da ...,166506
8,21,Valor/código numérico de Código da Unidade da ...,165756
9,43,Valor/código numérico de Código da Unidade da ...,159919


### `SG_UF_PROVA`

Sigla da Unidade da Federação da aplicação da prova

In [167]:
plotar_distribuicao_coluna('SG_UF_PROVA')

SG_UF_PROVA: 27 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,SP,SP,590767
1,MG,MG,358575
2,BA,BA,324268
3,RJ,RJ,282296
4,CE,CE,241960
5,PA,PA,229162
6,PE,PE,218859
7,PR,PR,166506
8,MA,MA,165756
9,RS,RS,159919


### `Q001`

Até que série seu pai, ou o homem responsável por você, estudou?

In [168]:
plotar_distribuicao_coluna('Q001')

Q001: 8 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Nunca estudou.,174712
1,B,Não completou a 4ª série/5º ano do Ensino Fund...,700175
2,C,"Completou a 4ª série/5º ano, mas não completou...",513213
3,D,Completou a 8ª série/9º ano do Ensino Fundamen...,438139
4,E,"Completou o Ensino Médio, mas não completou a ...",1114164
5,F,"Completou a Faculdade, mas não completou a Pós...",333981
6,G,Completou a Pós-graduação.,256045
7,H,Não sei.,403526


Maior frequência em Q001: 1,114,164


,label,significado,quantidade_registros
4,E,"Completou o Ensino Médio, mas não completou a ...",1114164


Menor frequência em Q001: 174,712


,label,significado,quantidade_registros
0,A,Nunca estudou.,174712


### `Q002`

Até que série sua mãe, ou a mulher responsável por você, estudou?

In [169]:
plotar_distribuicao_coluna('Q002')

Q002: 8 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Nunca estudou.,110831
1,B,Não completou a 4ª série/5º ano do Ensino Fund...,505525
2,C,"Completou a 4ª série/5º ano, mas não completou...",437022
3,D,Completou a 8ª série/9º ano do Ensino Fundamen...,473854
4,E,"Completou o Ensino Médio, mas não completou a ...",1377605
5,F,"Completou a Faculdade, mas não completou a Pós...",455268
6,G,Completou a Pós-graduação.,441893
7,H,Não sei.,131957


Maior frequência em Q002: 1,377,605


,label,significado,quantidade_registros
4,E,"Completou o Ensino Médio, mas não completou a ...",1377605


Menor frequência em Q002: 110,831


,label,significado,quantidade_registros
0,A,Nunca estudou.,110831


### `Q003`

A partir da apresentação de algumas ocupações divididas em grupos ordenados, indique o grupo que contempla a ocupação mais próxima da ocupação do seu pai ou do homem responsável por você. (Se ele não estiver trabalhando, escolha uma ocupação pensando no último trabalho dele).

In [170]:
plotar_distribuicao_coluna('Q003')

Q003: 6 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,"Grupo 1: Lavrador, agricultor sem empregados, ...",750778
1,B,"Grupo 2: Diarista, empregado doméstico, cuidad...",790173
2,C,"Grupo 3: Padeiro, cozinheiro industrial ou em ...",911756
3,D,Grupo 4: Professor (de ensino fundamental ou m...,706775
4,E,"Grupo 5: Médico, engenheiro, dentista, psicólo...",268192
5,F,Não sei.,506281


Maior frequência em Q003: 911,756


,label,significado,quantidade_registros
2,C,"Grupo 3: Padeiro, cozinheiro industrial ou em ...",911756


Menor frequência em Q003: 268,192


,label,significado,quantidade_registros
4,E,"Grupo 5: Médico, engenheiro, dentista, psicólo...",268192


### `Q004`

A partir da apresentação de algumas ocupações divididas em grupos ordenados, indique o grupo que contempla a ocupação mais próxima da ocupação da sua mãe ou da mulher responsável por você. (Se ela não estiver trabalhando, escolha uma ocupação pensando no último trabalho dela).

In [171]:
plotar_distribuicao_coluna('Q004')

Q004: 6 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,"Grupo 1: Lavradora, agricultora sem empregados...",608127
1,B,"Grupo 2: Diarista, empregada doméstica, cuidad...",1632671
2,C,"Grupo 3: Padeira, cozinheira industrial ou em ...",247158
3,D,Grupo 4: Professora (de ensino fundamental ou ...,892420
4,E,"Grupo 5: Médica, engenheira, dentista, psicólo...",200584
5,F,Não sei.,352995


Maior frequência em Q004: 1,632,671


,label,significado,quantidade_registros
1,B,"Grupo 2: Diarista, empregada doméstica, cuidad...",1632671


Menor frequência em Q004: 200,584


,label,significado,quantidade_registros
4,E,"Grupo 5: Médica, engenheira, dentista, psicólo...",200584


### `Q005`

Incluindo você, quantas pessoas moram atualmente em sua residência?

In [172]:
plotar_distribuicao_coluna('Q005')

Q005: 20 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,1,"1, pois moro sozinho(a).",146875
1,2,2,567635
2,3,3,1086674
3,4,4,1247259
4,5,5,566325
5,6,6,194132
6,7,7,71049
7,8,8,29968
8,9,9,11192
9,10,10,6826


Maior frequência em Q005: 1,247,259


,label,significado,quantidade_registros
3,4,4,1247259


Menor frequência em Q005: 57


,label,significado,quantidade_registros
18,19,19,57


### `Q006`

Qual é a renda mensal de sua família? (Some a sua renda com a dos seus familiares.)

In [173]:
plotar_distribuicao_coluna('Q006')

Q006: 17 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Nenhuma Renda,268053
1,B,"Até R$ 1.320,00",1245271
2,C,"De R$ 1.320,01 até R$ 1.980,00.",650942
3,D,"De R$ 1.980,01 até R$ 2.640,00.",437366
4,E,"De R$ 2.640,01 até R$ 3.300,00.",293994
5,F,"De R$ 3.300,01 até R$ 3.960,00.",171344
6,G,"De R$ 3.960,01 até R$ 5.280,00.",261327
7,H,"De R$ 5.280,01 até R$ 6.600,00.",139279
8,I,"De R$ 6.600,01 até R$ 7.920,00.",85970
9,J,"De R$ 7.920,01 até R$ 9240,00.",75179


Maior frequência em Q006: 1,245,271


,label,significado,quantidade_registros
1,B,"Até R$ 1.320,00",1245271


Menor frequência em Q006: 33,626


,label,significado,quantidade_registros
12,M,"De R$ 11.880,01 até R$ 13.200,00.",33626


### `Q007`

Em sua residência trabalha empregado(a) doméstico(a)?

In [174]:
plotar_distribuicao_coluna('Q007')

Q007: 4 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3611179
1,B,"Sim, um ou dois dias por semana.",178524
2,C,"Sim, três ou quatro dias por semana.",39882
3,D,"Sim, pelo menos cinco dias por semana.",104370


Maior frequência em Q007: 3,611,179


,label,significado,quantidade_registros
0,A,Não.,3611179


Menor frequência em Q007: 39,882


,label,significado,quantidade_registros
2,C,"Sim, três ou quatro dias por semana.",39882


### `Q008`

Na sua residência tem banheiro?

In [175]:
plotar_distribuicao_coluna('Q008')

Q008: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,37878
1,B,"Sim, um.",2647503
2,C,"Sim, dois.",855720
3,D,"Sim, três.",253087
4,E,"Sim, quatro ou mais.",139767


Maior frequência em Q008: 2,647,503


,label,significado,quantidade_registros
1,B,"Sim, um.",2647503


Menor frequência em Q008: 37,878


,label,significado,quantidade_registros
0,A,Não.,37878


### `Q009`

Na sua residência tem quartos para dormir?

In [176]:
plotar_distribuicao_coluna('Q009')

Q009: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,34469
1,B,"Sim, um.",560699
2,C,"Sim, dois.",1968685
3,D,"Sim, três.",1167141
4,E,"Sim, quatro ou mais.",202961


Maior frequência em Q009: 1,968,685


,label,significado,quantidade_registros
2,C,"Sim, dois.",1968685


Menor frequência em Q009: 34,469


,label,significado,quantidade_registros
0,A,Não.,34469


### `Q010`

Na sua residência tem carro?

In [177]:
plotar_distribuicao_coluna('Q010')

Q010: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,2167356
1,B,"Sim, um.",1392596
2,C,"Sim, dois.",323334
3,D,"Sim, três.",41185
4,E,"Sim, quatro ou mais.",9484


Maior frequência em Q010: 2,167,356


,label,significado,quantidade_registros
0,A,Não.,2167356


Menor frequência em Q010: 9,484


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",9484


### `Q011`

Na sua residência tem motocicleta?

In [178]:
plotar_distribuicao_coluna('Q011')

Q011: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,2983690
1,B,"Sim, uma.",842923
2,C,"Sim, duas.",95329
3,D,"Sim, três.",10022
4,E,"Sim, quatro ou mais.",1991


Maior frequência em Q011: 2,983,690


,label,significado,quantidade_registros
0,A,Não.,2983690


Menor frequência em Q011: 1,991


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",1991


### `Q012`

Na sua residência tem geladeira?

In [179]:
plotar_distribuicao_coluna('Q012')

Q012: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,75430
1,B,"Sim, uma.",3653942
2,C,"Sim, duas.",186456
3,D,"Sim, três.",14986
4,E,"Sim, quatro ou mais.",3141


Maior frequência em Q012: 3,653,942


,label,significado,quantidade_registros
1,B,"Sim, uma.",3653942


Menor frequência em Q012: 3,141


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",3141


### `Q013`

Na sua residência tem freezer (independente ou segunda porta da geladeira)?

In [180]:
plotar_distribuicao_coluna('Q013')

Q013: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,2217219
1,B,"Sim, um.",1580612
2,C,"Sim, dois.",117473
3,D,"Sim, três.",15112
4,E,"Sim, quatro ou mais.",3539


Maior frequência em Q013: 2,217,219


,label,significado,quantidade_registros
0,A,Não.,2217219


Menor frequência em Q013: 3,539


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",3539


### `Q014`

Na sua residência tem máquina de lavar roupa? (o tanquinho NÃO deve ser considerado)

In [181]:
plotar_distribuicao_coluna('Q014')

Q014: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,1517308
1,B,"Sim, uma.",2368207
2,C,"Sim, duas.",46133
3,D,"Sim, três.",1863
4,E,"Sim, quatro ou mais.",444


Maior frequência em Q014: 2,368,207


,label,significado,quantidade_registros
1,B,"Sim, uma.",2368207


Menor frequência em Q014: 444


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",444


### `Q015`

Na sua residência tem máquina de secar roupa (independente ou em conjunto com a máquina de lavar roupa)?

In [182]:
plotar_distribuicao_coluna('Q015')

Q015: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3438821
1,B,"Sim, uma.",488250
2,C,"Sim, duas.",6110
3,D,"Sim, três.",501
4,E,"Sim, quatro ou mais.",273


Maior frequência em Q015: 3,438,821


,label,significado,quantidade_registros
0,A,Não.,3438821


Menor frequência em Q015: 273


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",273


### `Q016`

Na sua residência tem forno micro-ondas?

In [183]:
plotar_distribuicao_coluna('Q016')

Q016: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,2064853
1,B,"Sim, um.",1838403
2,C,"Sim, dois.",28698
3,D,"Sim, três.",1515
4,E,"Sim, quatro ou mais.",486


Maior frequência em Q016: 2,064,853


,label,significado,quantidade_registros
0,A,Não.,2064853


Menor frequência em Q016: 486


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",486


### `Q017`

Na sua residência tem máquina de lavar louça?

In [184]:
plotar_distribuicao_coluna('Q017')

Q017: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3803308
1,B,"Sim, uma.",127927
2,C,"Sim, duas.",2180
3,D,"Sim, três.",280
4,E,"Sim, quatro ou mais.",260


Maior frequência em Q017: 3,803,308


,label,significado,quantidade_registros
0,A,Não.,3803308


Menor frequência em Q017: 260


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",260


### `Q018`

Na sua residência tem aspirador de pó?

In [185]:
plotar_distribuicao_coluna('Q018')

Q018: 2 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3032209
1,B,Sim.,901746


Maior frequência em Q018: 3,032,209


,label,significado,quantidade_registros
0,A,Não.,3032209


Menor frequência em Q018: 901,746


,label,significado,quantidade_registros
1,B,Sim.,901746


### `Q019`

Na sua residência tem televisão em cores?

In [186]:
plotar_distribuicao_coluna('Q019')

Q019: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,305208
1,B,"Sim, uma.",2607933
2,C,"Sim, duas.",661090
3,D,"Sim, três.",241516
4,E,"Sim, quatro ou mais.",118208


Maior frequência em Q019: 2,607,933


,label,significado,quantidade_registros
1,B,"Sim, uma.",2607933


Menor frequência em Q019: 118,208


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",118208


### `Q020`

Na sua residência tem aparelho de DVD?

In [187]:
plotar_distribuicao_coluna('Q020')

Q020: 2 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3430585
1,B,Sim.,503370


Maior frequência em Q020: 3,430,585


,label,significado,quantidade_registros
0,A,Não.,3430585


Menor frequência em Q020: 503,370


,label,significado,quantidade_registros
1,B,Sim.,503370


### `Q021`

Na sua residência tem TV por assinatura?

In [188]:
plotar_distribuicao_coluna('Q021')

Q021: 2 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3100005
1,B,Sim.,833950


Maior frequência em Q021: 3,100,005


,label,significado,quantidade_registros
0,A,Não.,3100005


Menor frequência em Q021: 833,950


,label,significado,quantidade_registros
1,B,Sim.,833950


### `Q022`

Na sua residência tem telefone celular?

In [189]:
plotar_distribuicao_coluna('Q022')

Q022: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,101584
1,B,"Sim, um.",764861
2,C,"Sim, dois.",1087753
3,D,"Sim, três.",1138368
4,E,"Sim, quatro ou mais.",841389


Maior frequência em Q022: 1,138,368


,label,significado,quantidade_registros
3,D,"Sim, três.",1138368


Menor frequência em Q022: 101,584


,label,significado,quantidade_registros
0,A,Não.,101584


### `Q023`

Na sua residência tem telefone fixo?

In [190]:
plotar_distribuicao_coluna('Q023')

Q023: 2 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,3579617
1,B,Sim.,354338


Maior frequência em Q023: 3,579,617


,label,significado,quantidade_registros
0,A,Não.,3579617


Menor frequência em Q023: 354,338


,label,significado,quantidade_registros
1,B,Sim.,354338


### `Q024`

Na sua residência tem computador?

In [191]:
plotar_distribuicao_coluna('Q024')

Q024: 5 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,2041637
1,B,"Sim, um.",1362634
2,C,"Sim, dois.",341120
3,D,"Sim, três.",126296
4,E,"Sim, quatro ou mais.",62268


Maior frequência em Q024: 2,041,637


,label,significado,quantidade_registros
0,A,Não.,2041637


Menor frequência em Q024: 62,268


,label,significado,quantidade_registros
4,E,"Sim, quatro ou mais.",62268


### `Q025`

Na sua residência tem acesso à Internet?

In [192]:
plotar_distribuicao_coluna('Q025')

Q025: 2 labels observados; ausentes: 0


,label,significado,quantidade_registros
0,A,Não.,375504
1,B,Sim.,3558451


Maior frequência em Q025: 3,558,451


,label,significado,quantidade_registros
1,B,Sim.,3558451


Menor frequência em Q025: 375,504


,label,significado,quantidade_registros
0,A,Não.,375504


### `NU_NOTA_MT`

Nota da prova de Matemática

In [193]:
plotar_distribuicao_coluna('NU_NOTA_MT')

Resumo de NU_NOTA_MT (variável numérica contínua):


,indicador,valor
0,menor nota,0.0
1,maior nota,958.6
2,valores distintos observados,6058.0
3,registros com nota,2692427.0
4,registros sem nota,1241528.0


In [ ]:
# Tabela de decisão preliminar das features, conforme a seleção definida pela equipe.
import pandas as pd

descricoes_tabela_features = {'CO_MUNICIPIO_ESC': 'Código do município da escola',
 'CO_MUNICIPIO_PROVA': 'Código do município da aplicação da prova',
 'CO_UF_ESC': 'Código da Unidade da Federação da escola',
 'CO_UF_PROVA': 'Código da Unidade da Federação da aplicação da prova',
 'IN_TREINEIRO': 'Indica se o inscrito fez a prova com intuito de apenas treinar seus '
                 'conhecimentos3',
 'NO_MUNICIPIO_ESC': 'Nome do município da escola',
 'NO_MUNICIPIO_PROVA': 'Nome do município da aplicação da prova',
 'NU_NOTA_MT': 'Nota da prova de Matemática',
 'Q001': 'Até que série seu pai, ou o homem responsável por você, estudou?',
 'Q002': 'Até que série sua mãe, ou a mulher responsável por você, estudou?',
 'Q003': 'A partir da apresentação de algumas ocupações divididas em grupos ordenados, indique o '
         'grupo que contempla a ocupação mais próxima da ocupação do seu pai ou do homem '
         'responsável por você. (Se ele não estiver trabalhando, escolha uma ocupação pensando no '
         'último trabalho dele).',
 'Q004': 'A partir da apresentação de algumas ocupações divididas em grupos ordenados, indique o '
         'grupo que contempla a ocupação mais próxima da ocupação da sua mãe ou da mulher '
         'responsável por você. (Se ela não estiver trabalhando, escolha uma ocupação pensando no '
         'último trabalho dela).',
 'Q005': 'Incluindo você, quantas pessoas moram atualmente em sua residência?',
 'Q006': 'Qual é a renda mensal de sua família? (Some a sua renda com a dos seus familiares.)',
 'Q007': 'Em sua residência trabalha empregado(a) doméstico(a)?',
 'Q008': 'Na sua residência tem banheiro?',
 'Q009': 'Na sua residência tem quartos para dormir?',
 'Q010': 'Na sua residência tem carro?',
 'Q011': 'Na sua residência tem motocicleta?',
 'Q012': 'Na sua residência tem geladeira?',
 'Q013': 'Na sua residência tem freezer (independente ou segunda porta da geladeira)?',
 'Q014': 'Na sua residência tem máquina de lavar roupa? (o tanquinho NÃO deve ser considerado)',
 'Q015': 'Na sua residência tem máquina de secar roupa (independente ou em conjunto com a máquina '
         'de lavar roupa)?',
 'Q016': 'Na sua residência tem forno micro-ondas?',
 'Q017': 'Na sua residência tem máquina de lavar louça?',
 'Q018': 'Na sua residência tem aspirador de pó?',
 'Q019': 'Na sua residência tem televisão em cores?',
 'Q020': 'Na sua residência tem aparelho de DVD?',
 'Q021': 'Na sua residência tem TV por assinatura?',
 'Q022': 'Na sua residência tem telefone celular?',
 'Q023': 'Na sua residência tem telefone fixo?',
 'Q024': 'Na sua residência tem computador?',
 'Q025': 'Na sua residência tem acesso à Internet?',
 'SG_UF_ESC': 'Sigla da Unidade da Federação da escola',
 'SG_UF_PROVA': 'Sigla da Unidade da Federação da aplicação da prova',
 'TP_ANO_CONCLUIU': 'Ano de Conclusão do Ensino Médio',
 'TP_COR_RACA': 'Cor/raça',
 'TP_DEPENDENCIA_ADM_ESC': 'Dependência administrativa (Escola)',
 'TP_ENSINO': 'Tipo de instituição que concluiu ou concluirá o Ensino Médio',
 'TP_ESCOLA': 'Tipo de escola do Ensino Médio',
 'TP_ESTADO_CIVIL': 'Estado Civil',
 'TP_FAIXA_ETARIA': 'Faixa etária2',
 'TP_LOCALIZACAO_ESC': 'Localização (Escola)',
 'TP_NACIONALIDADE': 'Nacionalidade',
 'TP_SEXO': 'Sexo',
 'TP_SIT_FUNC_ESC': 'Situação de funcionamento (Escola)',
 'TP_ST_CONCLUSAO': 'Situação de conclusão do Ensino Médio'}
# Esta célula documenta a decisão; não filtra participantes nem altera o pipeline de treino.
linhas_decisao_features = [('Sim', 1, 'TP_FAIXA_ETARIA', '14', 20, 'Representa o perfil etário do participante e pode contextualizar sua trajetória escolar.'), ('Sim', 2, 'TP_SEXO', 'M', 2, 'Característica demográfica disponível antes da prova; permite investigar diferenças de desempenho entre grupos sem inferir causalidade.'), ('Sim', 3, 'TP_ESTADO_CIVIL', '2', 5, 'Característica demográfica de perfil, com poucas categorias interpretáveis.'), ('Sim', 4, 'TP_COR_RACA', '1', 6, 'Característica sociodemográfica relevante para descrever grupos e investigar desigualdades; interpretar com cuidado e sem causalidade.'), ('Não', 5, 'TP_NACIONALIDADE', '1', 5, 'Decisão de escopo para a seleção inicial: tende a acrescentar pouca diferenciação diante da concentração na categoria brasileira. Não significa que seja impossível testar seu valor depois.'), ('Sim', 6, 'TP_ST_CONCLUSAO', '1', 4, 'Resume se o participante já concluiu, está concluindo ou não está cursando o Ensino Médio; descreve sua trajetória escolar.'), ('Não', 7, 'TP_ANO_CONCLUIU', '17', 18, 'Na seleção inicial, foi deixada de fora por sobrepor parte da informação de TP_ST_CONCLUSAO; além disso, 0 é “não informado” e 17 agrupa todos os anos anteriores a 2007. Pode ser testada depois como hipótese adicional.'), ('Sim', 8, 'TP_ESCOLA', '1', 3, 'Distingue o tipo de escola de Ensino Médio (pública/privada/não respondeu), um aspecto do contexto escolar.'), ('Sim', 9, 'TP_ENSINO', '1.0', 2, 'Selecionada pela equipe, mas exige cautela: 65,96% de ausências na base completa e 61,66% entre participantes com nota observada. Inspecionar a causa e comparar o tratamento das ausências; não imputar automaticamente.'), ('Sim', 10, 'IN_TREINEIRO', '0', 2, 'Pode ser usada como feature porque identifica uma condição do participante conhecida na inscrição. Nesta base, está perfeitamente associada a TP_ST_CONCLUSAO (V de Cramér = 1,00), então compare se ambas acrescentam informação. Manter os participantes treineiros na base, independentemente desta feature.'), ('Não', 11, 'CO_MUNICIPIO_ESC', '2304400.0', 5528, '75,64% de ausências na base completa (73,10% entre participantes com nota observada), além de alta cardinalidade. Código e nome do município escolar são redundantes e não indicam residência.'), ('Não', 12, 'NO_MUNICIPIO_ESC', 'Fortaleza', 5261, '75,64% de ausências na base completa (73,10% entre participantes com nota observada), além de alta cardinalidade. Nome repete o código do município escolar, que não equivale à residência.'), ('Não', 13, 'CO_UF_ESC', '23.0', 27, '75,64% de ausências na base completa (73,10% entre participantes com nota observada) e duplicidade com a sigla SG_UF_ESC.'), ('Não', 14, 'SG_UF_ESC', 'CE', 27, '75,64% de ausências na base completa (73,10% entre participantes com nota observada) e duplicidade com o código CO_UF_ESC.'), ('Não', 15, 'TP_DEPENDENCIA_ADM_ESC', '2.0', 4, '75,64% de ausências na base completa (73,10% entre participantes com nota observada). Pode informar o contexto escolar, mas a baixa cobertura motivou deixá-la fora da primeira configuração; pode ser testada depois com tratamento explícito.'), ('Não', 16, 'TP_LOCALIZACAO_ESC', '1.0', 2, '75,64% de ausências na base completa (73,10% entre participantes com nota observada). Pode informar o contexto escolar, mas a baixa cobertura motivou deixá-la fora da primeira configuração; pode ser testada depois com tratamento explícito.'), ('Não', 17, 'TP_SIT_FUNC_ESC', '1.0', 4, '75,64% de ausências na base completa (73,10% entre participantes com nota observada), além de ser informação indireta sobre a trajetória individual; deixada fora da primeira configuração.'), ('Não', 18, 'CO_MUNICIPIO_PROVA', '5300108', 1750, 'Local da aplicação da prova não equivale ao município de residência; possui muitas categorias e pode identificar sobretudo o local do exame.'), ('Não', 19, 'NO_MUNICIPIO_PROVA', 'Brasília', 1715, 'Local da aplicação não é necessariamente o local onde o participante mora; também repete a informação do código do município da prova.'), ('Não', 20, 'CO_UF_PROVA', '53', 27, 'Representa a UF de aplicação, não necessariamente a residência; foi excluída no conjunto inicial e duplica a informação de SG_UF_PROVA.'), ('Não', 21, 'SG_UF_PROVA', 'DF', 27, 'Representa a UF de aplicação, não necessariamente a residência; foi excluída no conjunto inicial e duplica a informação de CO_UF_PROVA.'), ('Sim', 22, 'Q001', 'A', 8, 'Escolaridade do pai ou homem responsável; aproxima o contexto educacional familiar. Tratar categorias segundo o dicionário, sem presumir distâncias iguais entre códigos.'), ('Sim', 23, 'Q002', 'F', 8, 'Escolaridade da mãe ou mulher responsável; aproxima o contexto educacional familiar. Tratar categorias segundo o dicionário.'), ('Sim', 24, 'Q003', 'E', 6, 'Grupo ocupacional do pai ou homem responsável; pode caracterizar o contexto socioeconômico familiar. Os códigos são categorias, não uma escala numérica.'), ('Sim', 25, 'Q004', 'D', 6, 'Grupo ocupacional da mãe ou mulher responsável; pode caracterizar o contexto socioeconômico familiar. Os códigos são categorias, não uma escala numérica.'), ('Sim', 26, 'Q005', '5', 20, 'Número de pessoas na residência; contextualiza o tamanho do domicílio e pode ajudar a interpretar recursos familiares.'), ('Sim', 27, 'Q006', 'F', 17, 'Faixa de renda familiar; indicador direto do contexto socioeconômico. As faixas são categóricas/ordinais, não valores monetários exatos.'), ('Sim', 28, 'Q007', 'C', 4, 'Presença de empregado(a) doméstico(a); indicador de recursos e condições do domicílio.'), ('Sim', 29, 'Q008', 'C', 5, 'Existência de banheiro na residência; indicador de condições domiciliares.'), ('Sim', 30, 'Q009', 'D', 5, 'Quantidade de quartos para dormir; indicador de estrutura domiciliar.'), ('Sim', 31, 'Q010', 'C', 5, 'Posse de carro; indicador de bens e recursos do domicílio.'), ('Sim', 32, 'Q011', 'D', 5, 'Posse de motocicleta; indicador de bens e recursos do domicílio.'), ('Sim', 33, 'Q012', 'C', 5, 'Posse de geladeira; indicador de bens e infraestrutura doméstica.'), ('Sim', 34, 'Q013', 'B', 5, 'Posse de freezer; indicador de bens e recursos do domicílio.'), ('Sim', 35, 'Q014', 'B', 5, 'Posse de máquina de lavar roupa; indicador de bens e recursos do domicílio.'), ('Sim', 36, 'Q015', 'D', 5, 'Posse de máquina de secar roupa; indicador de bens e recursos do domicílio.'), ('Sim', 37, 'Q016', 'C', 5, 'Posse de forno micro-ondas; indicador de bens e recursos do domicílio.'), ('Sim', 38, 'Q017', 'C', 5, 'Posse de máquina de lavar louça; indicador de bens e recursos do domicílio.'), ('Sim', 39, 'Q018', 'B', 2, 'Posse de aspirador de pó; indicador de bens do domicílio.'), ('Sim', 40, 'Q019', 'B', 5, 'Quantidade de televisores em cores; indicador de bens do domicílio.'), ('Sim', 41, 'Q020', 'A', 2, 'Posse de aparelho de DVD; indicador de bens do domicílio.'), ('Sim', 42, 'Q021', 'B', 2, 'Acesso a TV por assinatura; indicador de recursos/acesso a serviços no domicílio.'), ('Sim', 43, 'Q022', 'B', 5, 'Quantidade de celulares; indicador de bens e acesso à comunicação.'), ('Sim', 44, 'Q023', 'A', 2, 'Posse de telefone fixo; indicador de bens e acesso à comunicação.'), ('Sim', 45, 'Q024', 'A', 5, 'Quantidade de computadores; indicador de recursos que também podem apoiar o acesso a estudos.'), ('Sim', 46, 'Q025', 'B', 2, 'Acesso à Internet; indicador de conectividade e possibilidade de acesso a recursos de estudo.'), ('Valor Y', 47, 'NU_NOTA_MT', '363.2', 6058, 'Alvo contínuo da regressão (nota de Matemática); é a variável a prever, não uma feature de entrada.')]
colunas_decisao_features = [
    "Faz sentido?", "ordem", "nome_da_coluna", "exemplo",
    "qtd_valores_distintos", "justificativa",
]
df_decisao_features = pd.DataFrame(linhas_decisao_features, columns=colunas_decisao_features)
df_decisao_features.insert(
    5,
    "descricao",
    df_decisao_features["nome_da_coluna"].map(descricoes_tabela_features),
)
df_decisao_features = df_decisao_features[[
    "Faz sentido?", "ordem", "nome_da_coluna", "exemplo",
    "qtd_valores_distintos", "descricao", "justificativa",
]]
display(df_decisao_features)
